## Task 6 - Judge vs. Human Evaluation

To evaluate the LLM judge, I compared its outputs against the manually labeled rows in the original `data/assignment_01.csv` file.  
In practice, the useful comparison set was the subset of rows that already had labels filled in by a human (myself).

I compared two judge setups:
- a smaller judge model (***google/gemma-2-9b-it-fast***)
- a larger judge model (***Qwen/Qwen3-30B-A3B-Instruct-2507***)

---

## 1. Sanity check
Ran `uv run python -m src.judge_main --take-rows 5` - Results were decent.

## 2. Full run
#### Small model
- Ran: `uv run python -m src.judge_main`
- Saved file: `data/judge_results/assignment_01_judged.csv`
#### Large model: 
- Ran: `uv run python -m src.judge_main --experiment larger_model --llm-model Qwen/Qwen3-30B-A3B-Instruct-2507`
- Saved file: `data/judge_results/larger_model_judged.csv` 


## 3. Compare to human evaluation

To evaluate the judge, I compared its verdicts against my manual labels from Task 3 on the subset of rows that I had already annotated by hand.

For each criterion, I computed an **agreement rate**:

**agreement rate = number of rows where the judge verdict matched the human verdict / number of human-labeled rows**

Using the stronger judge setup, I observed the following agreement rates on the human-labeled subset:

- Fluency - 15 / 15 = 100.0%
- Grammar - 15 / 15 = 100.0%
- Tone - 15 / 15 = 100.0%
- Length - 14 / 15 = 93.3%
- Grounding - 6 / 15 = 40.0%
- Final score - 14 / 15 = 93.3%

### Where they agreed

The judge agreed strongly with the human evaluation on:

- **fluency**
- **grammar**
- **tone**

These criteria were relatively easy for the judge because most of the generated descriptions were polished, grammatically correct, and written in a reasonable e-commerce tone.

The judge also performed well on:

- **length**
- **final score**

although it still made at least one error on length.

### Where they diverged

The biggest disagreement was on **grounding**.

The human evaluation was stricter about mild unsupported claims such as:

- premium
- durable
- built to last
- perfect for work or play
- enhanced durability

The judge often treated these as acceptable marketing language and still assigned `GOOD`, while the human labels more often assigned `OK`.

This means the judge was generally too lenient on soft exaggeration and product-quality language.

There was also a smaller divergence on **length**.  
Even though length is a simple numeric criterion, the judge made at least one incorrect decision despite the explicit rubric.

### Why they diverged

The main reason for disagreement is that the judge seems influenced by the overall quality of the writing. When a description sounds polished and professional, the judge becomes more forgiving, especially on grounding.

This creates a kind of **halo effect**:

- good fluency and strong tone make the judge more likely to rate grounding as `GOOD`
- mild unsupported claims are interpreted as harmless marketing language instead of rubric violations

In contrast, human evaluation was more sensitive to the distinction between:

- acceptable mild interpretation
- unsupported quality claims

---

## 4. Criterion-by-Criterion Judging

Yes - isolating criteria **did change the results**.

Compared with the original all-at-once judge, the isolated runs were less influenced by the model's overall impression of the description. In the all-at-once setup, a polished description often received generous scores across several criteria, especially grounding. In the criterion-by-criterion setup, the judge was forced to focus on one question at a time, which reduced some of that spillover.

### Why isolating criteria can lead to different outcomes

There are a few likely reasons:

- **Reduced cross-criterion contamination** - when the judge evaluates all criteria at once, strong fluency or polished tone can make it more forgiving on grounding
- **Narrower attention** - a criterion-only prompt encourages the model to look only for evidence relevant to that specific rubric
- **Less halo effect** - the judge is less likely to think "this sounds good overall, so everything must be good"
- **Simpler decision boundary** - deciding one label for one criterion is cognitively easier than balancing five criteria plus a final score in the same call

### Did agreement with human scores improve?

**Partly yes.**  
The improvement was not uniform across criteria.

#### Fluency
Agreement **appeared to improve**.

The `fluency_only` run was more focused on sentence flow, awkward phrasing, and repetition. It caught at least some fluency issues more directly, such as the awkward Fitbit phrasing, without mixing in grounding or tone logic. That makes it closer to how I applied the human rubric.

#### Grammar
Agreement also **appeared to improve**.

The isolated grammar run was better at looking specifically for visible language issues. It correctly downgraded rows with awkward or malformed wording, such as Fitbit and Hydro Flask. It still sometimes treated wording problems as grammar, but overall it looked more aligned with human judgment than the all-at-once version.

#### Grounding
Agreement **improved slightly, but not enough**.

This was still the hardest criterion. The isolated grounding judge did better on some cases, for example by downgrading Dell and Fitbit. However, it still remained too lenient in many rows. It often accepted soft unsupported marketing claims such as:

- premium
- durable
- built to last
- high praise

as fully grounded, even when my human label was stricter. So grounding improved somewhat, but it was still the weakest area.

#### Length
Agreement did **not improve reliably**.

In fact, the `length_only` run still misapplied the rubric in multiple rows. Several descriptions that were clearly within the `50-90 words` range were labeled `OK` instead of `GOOD`. Since length is a strict numeric rule, this criterion should really be calculated in code rather than judged by an LLM.

#### Tone
Tone changed only slightly under criterion-by-criterion judging.

The isolated tone run appeared somewhat more careful in a few borderline cases and downgraded some rows from `GOOD` to `OK`, especially when the tone was more neutral or slightly polished rather than strongly persuasive. However, tone was already one of the more stable criteria in the original all-at-once setup, so the improvement was smaller than for fluency or grammar.

Overall, isolating tone helped a little, but not dramatically. The judge still tended to rate many descriptions as `GOOD`, and some of its explanations were repetitive and generic rather than highly discriminating.

### Final conclusion

Criterion-by-criterion judging **did change the results**, and in general it made the judge more faithful to the intended rubric by reducing interference between criteria.

However, the benefit was **uneven**:

- **Fluency** - improved
- **Grammar** - improved
- **Grounding** - improved slightly, but still too lenient
- **Length** - did not improve reliably and should be rule-based instead
- **Tone** — slight improvement

So my conclusion is:

> Isolating criteria helps because it reduces overall-impression bias and forces the judge to focus on one rubric at a time. This improved agreement with my human scores for fluency and grammar, helped somewhat for grounding, but did not solve grounding fully and did not work well for length.

### Practical takeaway

A stronger production-style setup would be:

- use **rule-based checks** for length, latency, and cost
- use **criterion-by-criterion LLM judging** for fluency, grammar, tone, and grounding
- keep **human review** for grounding and borderline cases

---

## 5. Analysis

### a. Practical trade-offs between human evaluation and LLM-as-a-judge
#### Human evaluation

Human evaluation is stronger in **accuracy on subtle cases**, especially for criteria like **grounding** and sometimes **tone**. A person can better notice when a description sounds polished but still makes claims that are not fully supported by the source product data. In my results, this was important because the LLM judge often accepted soft marketing exaggerations such as `premium`, `durable`, or `built to last` more readily than I did in the human evaluation.

The main weakness of human evaluation is that it is **slow, expensive, and hard to scale**. Evaluating a handful of examples is manageable, but evaluating thousands of descriptions every day would require a large amount of manual effort. Human judgment can also be less consistent over time, especially if multiple reviewers are involved or if the same reviewer becomes more or less strict across sessions.

#### LLM-as-a-judge

LLM-as-a-judge is much stronger in **speed, scale, and consistency of format**. Once the judge is set up, it can score many descriptions quickly and produce structured outputs for every criterion. It is especially useful when there are many outputs to screen and when a uniform format is needed for downstream analysis.

However, the judge is not always as accurate as a human, especially on **borderline cases**. In my experiments, the judge was often too lenient on grounding. It sometimes treated unsupported quality language as acceptable marketing phrasing instead of reducing the grounding score. This means that the judge can be **systematically biased**, even when its explanations sound confident and well written.

So the practical trade-off is:

- **Humans** are slower and more expensive, but stronger on nuanced correctness
- **LLM judges** are faster and more scalable, but weaker on subtle reasoning and rubric strictness

---

### b. Recommended approach for a production system generating thousands of descriptions daily

For a production system that generates thousands of descriptions per day, I would **not** recommend relying only on human evaluation. That would be too slow and too expensive.

I also would **not** recommend relying only on an LLM judge with no safeguards, because my results showed that the judge can be too permissive, especially on grounding.

The best production approach would be a **hybrid evaluation pipeline**:

#### 1. Rule-based checks for objective criteria

Use deterministic code for criteria that do not require judgment, such as:

- word count / length
- latency
- token usage
- cost
- missing fields
- output format

These checks are cheap, fast, and perfectly consistent.

#### 2. LLM-as-a-judge for scalable semantic review

Use an LLM judge for criteria that require interpretation, such as:

- fluency
- grammar
- tone
- grounding

This gives scalable first-pass evaluation across large volumes of generated descriptions.

#### 3. Human review for a smaller subset

Keep human evaluation for:

- random quality audits
- borderline cases
- high-value products
- cases where the judge is uncertain
- categories where grounding is especially important

This is the most practical compromise because it keeps the system scalable while still protecting quality.

---

## Final recommendation

If I had to choose one production strategy, I would recommend:

- **automatic rule-based checks** for objective metrics
- **LLM-as-a-judge** for large-scale semantic evaluation
- **human spot checks and escalation review** for sensitive or borderline cases

This approach gives the best balance between:

- **cost**
- **scale**
- **consistency**
- **accuracy**

In other words, human evaluation is the gold standard for nuanced quality, but it does not scale well. LLM-as-a-judge scales well, but it should be supervised rather than trusted blindly. A hybrid system is therefore the most practical production solution.